### 1. Define what “right” means first.



`Our` merge logic has these invariants:

- Within one microbatch: for a given (wiki, title), only the row with the max timestamp survives dedup, regardless of input order.
- Across batches: a new row for an existing key only overwrites if its timestamp is strictly newer.
- A late/out-of-order row (older timestamp arriving after a newer one was already merged) must be a no-op, not overwrite.
type = 'new' matching an existing key is currently silently dropped (no insert, no update) — confirm that’s actually what you want.

In [0]:
%run ./params_config

### 2. Unit-test merge_to_silver directly with synthetic input, 
bypassing the stream entirely — this is the fastest way to prove the logic, and matches your CLAUDE.md pattern of testing each function before moving on. Run against a throwaway table so it doesn’t touch real data:

In [0]:
test_silver_table = f"{catalog}.{silver_schema}.wikimedia_silver_test"
spark.sql(f"CREATE OR REPLACE TABLE {test_silver_table} (id long, title string, comment string, title_url string, timestamp timestamp, user string, wiki string)")

def merge_to_silver_test(microBatchDF, batch_id):
    # same logic as merge_to_silver, but targets test_silver_table
    w = Window.partitionBy("wiki", "title").orderBy(col("timestamp").desc())
    deduped = microBatchDF.withColumn("rn", row_number().over(w)).filter("rn = 1").drop("rn")
    deduped.createOrReplaceTempView("wikimedia_test")
    spark.sql(f"""
        MERGE INTO {test_silver_table} t
        USING wikimedia_test s
        ON t.wiki = s.wiki AND t.title = s.title
        WHEN MATCHED AND s.type = 'edit' AND t.timestamp < s.timestamp THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:

# Case A: three events, same key, out of order within one microbatch -> highest timestamp wins
batch1 = spark.createDataFrame([
    (1, "Dog", "c1", "u1", 100, "new",  "a", "en"),
    (2, "Dog", "c2", "u1", 300, "edit", "b", "en"),
    (3, "Dog", "c3", "u1", 200, "edit", "c", "en"),], 

["id","title","comment","title_url","timestamp","type","user","wiki"]).withColumn("timestamp", col("timestamp").cast("timestamp"))


merge_to_silver_test(batch1, 0)

r = spark.table(test_silver_table).filter("wiki='en' and title='Dog'").collect()

assert len(r) == 1 and r[0].id == 2, f"expected id=2, got {r}"

print("[test A] pass: dedup kept latest timestamp in microbatch")
display(spark.table(test_silver_table))

In [0]:



# Case B: later microbatch, newer edit -> should update
batch2 = spark.createDataFrame([
    (4, "Dog", "c4", "u1", 400, "edit", "d", "en"),
], ["id","title","comment","title_url","timestamp","type","user","wiki"]).withColumn("timestamp", col("timestamp").cast("timestamp"))

merge_to_silver_test(batch2, 1)

r = spark.table(test_silver_table).filter("wiki='en' and title='Dog'").collect()

assert r[0].id == 4, f"expected id=4 after newer edit, got {r}"

print("[test B] pass: newer edit updated the row")
display(spark.table(test_silver_table))

In [0]:


# Case C: late-arriving OLDER edit -> must be a no-op
batch3 = spark.createDataFrame([
    (5, "Dog", "c5", "u1", 250, "edit", "e", "en"),
], ["id","title","comment","title_url","timestamp","type","user","wiki"]).withColumn("timestamp", col("timestamp").cast("timestamp"))

merge_to_silver_test(batch3, 2)

r = spark.table(test_silver_table).filter("wiki='en' and title='Dog'").collect()

assert r[0].id == 4, f"expected id=4 still (late event ignored), got {r}"

print("[test C] pass: late/older edit correctly ignored")

display(spark.table(test_silver_table))

In [0]:
spark.sql(f"DROP TABLE {test_silver_table}")